# Pipline
> Token IDs → Embedding layer → 1D convolution → ReLU → Masked global max pooling → Dropout  → Fully connected classification layer → Binary prediction

Can pre-learning semantic and sentiment information improve IMDb sentiment classification performance?

Do 50k unlabeled movie reviews offer any additional value?

# Work

## Embedding

[1]. Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). Learning word vectors
for sentiment analysis. Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human
Language Technologies, 142–150. https://ai.stanford.edu/~ang/papers/acl11-WordVectorsSentimentAnalysis.pdf

In [ ]:
#Hyperparameters

# Data
DATASET_NAME = "Kwaai/IMDB_Sentiment"
VALID_RATIO = 0.20

# Vocabulary
VOCAB_SIZE = 8_000
DROP_TOP_N = 50
PROTECTED_TOKENS = {
    "not", "no", "never", "neither", "nor",
    "but", "however", "although", "though", "yet",
    "good", "bad", "great", "best", "worst",
    "like", "love", "hate", "enjoy",
    "very", "really", "too", "so", "just",
    "more", "most", "less", "least",
    "hardly", "barely", "scarcely",
}
# Embedding model
EMBED_DIM = 50
EMBED_INIT_STD = 0.02

# Optimisation
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-3
BATCH_SIZE = 256
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-5
SENTIMENT_WEIGHT = 1.0
GRAD_CLIP = 5.0

# Reproducibility and output
SEED = 42
DEVICE = "auto"
NUM_WORKERS=8

In [ ]:
#import
import html
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(SEED)
device = resolve_device(DEVICE)

print("Device:", device)

c:\Users\lulu\anaconda3\envs\yolo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
#get_data
raw_dataset = load_dataset(
    DATASET_NAME,
    download_mode="force_redownload",
)

labelled_split = raw_dataset["train"].train_test_split(
    test_size=VALID_RATIO,
    seed=SEED,
    stratify_by_column="label",
)

train_dataset = labelled_split["train"]
valid_dataset = labelled_split["test"]
unlabelled_dataset = raw_dataset["unsupervised"]
test_dataset = raw_dataset["test"]

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 531921.07 examples/s]


In [ ]:
train_dataset.to_pandas()["label"].value_counts()

label
1    10000
0    10000
Name: count, dtype: int64

In [ ]:
valid_dataset.to_pandas()["label"].value_counts()

label
0    2500
1    2500
Name: count, dtype: int64

In [ ]:
test_dataset.to_pandas()["label"].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

word bank

In [ ]:
BREAK_PATTERN = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)
TOKEN_PATTERN = re.compile(
    r"[a-z]+(?:'[a-z]+)?|[!?]+|[:;=8xX][\-^']?[)(/\\dDpP]"
)


def tokenise(text):
    text = html.unescape(text)
    text = BREAK_PATTERN.sub(" ", text)
    return TOKEN_PATTERN.findall(text.lower())

In [ ]:
token_counts = Counter()

for text in tqdm(
    train_dataset["text"],
    desc=f"Building vocabulary from {len(train_dataset):,} train reviews",
):
    token_counts.update(tokenise(text))

ranked_tokens = token_counts.most_common()


def is_protected_token(token):
    return (
        token in PROTECTED_TOKENS
        or token.endswith("n't")
        or re.fullmatch(r"[!?]+", token) is not None
    )


ignored_tokens = [
    token
    for token, _ in ranked_tokens
    if not is_protected_token(token)
][:DROP_TOP_N]
ignored_token_set = set(ignored_tokens)

vocabulary_tokens = [
    token
    for token, _ in ranked_tokens
    if token not in ignored_token_set
][:VOCAB_SIZE]

if len(vocabulary_tokens) < VOCAB_SIZE:
    raise ValueError("The training corpus contains fewer tokens than VOCAB_SIZE.")

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

id_to_token = [PAD_TOKEN, UNK_TOKEN] + vocabulary_tokens
token_to_id = {
    token: token_id
    for token_id, token in enumerate(id_to_token)
}
TOTAL_VOCAB_SIZE = len(id_to_token)

print("Ignored top-frequency tokens:", ignored_tokens[:50])
print("Embedding matrix rows:", TOTAL_VOCAB_SIZE)

Building vocabulary from 20,000 train reviews: 100%|██████████| 20000/20000 [00:02<00:00, 8419.23it/s]

Ignored top-frequency tokens: ['the', 'and', 'a', 'of', 'to', 'is', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with', 'movie', 'film', 'on', 'you', 'his', 'are', 'have', 'he', 'be', 'one', 'all', 'at', 'by', 'an', 'they', 'who', 'from', 'her', 'or', 'about', "it's", 'out', 'if', 'has', 'there', 'some', 'what', 'when', 'up', 'even', 'time', 'she', 'my', 'would', 'which']
Embedding matrix rows: 8002


Encode documents

In [ ]:
def encode_documents(texts, description):
    encoded = []

    for text in tqdm(texts, desc=description):
        token_ids = [
            token_to_id[token]
            for token in tokenise(text)
            if token in token_to_id
        ]

        if not token_ids:
            token_ids = [UNK_ID]

        encoded.append(torch.tensor(token_ids, dtype=torch.long))

    return encoded


train_documents = encode_documents(
    train_dataset["text"],
    "Encoding labelled train reviews",
)
unlabelled_documents = encode_documents(
    unlabelled_dataset["text"],
    "Encoding unlabelled reviews",
)
train_labels = torch.tensor(
    train_dataset["label"],
    dtype=torch.long,
)
unlabelled_labels = torch.full(
    (len(unlabelled_documents),),
    -1,
    dtype=torch.long,
)

print("Labelled documents:", len(train_documents))
print("Unlabelled documents:", len(unlabelled_documents))

Encoding unlabelled reviews: 100%|██████████| 50000/50000 [00:04<00:00, 10836.67it/s]


Labelled documents: 20000
Unlabelled documents: 50000


In [ ]:
all_semantic_documents = train_documents + unlabelled_documents
all_semantic_labels = torch.cat(
    [train_labels, unlabelled_labels]
)

pretraining_specs = {
    "semantic_only": {
        "documents": train_documents,
        "labels": torch.full_like(train_labels, -1),
        "use_sentiment": False,
    },
    "semantic_sentiment": {
        "documents": train_documents,
        "labels": train_labels,
        "use_sentiment": True,
    },
    "semantic_sentiment_unlabeled": {
        "documents": all_semantic_documents,
        "labels": all_semantic_labels,
        "use_sentiment": True,
    },
}

group_summary = pd.DataFrame(
    [
        {
            "group": "random",
            "semantic_documents": 0,
            "sentiment_documents": 0,
        },
        {
            "group": "semantic_only",
            "semantic_documents": len(train_documents),
            "sentiment_documents": 0,
        },
        {
            "group": "semantic_sentiment",
            "semantic_documents": len(train_documents),
            "sentiment_documents": int((train_labels >= 0).sum()),
        },
        {
            "group": "semantic_sentiment_unlabeled",
            "semantic_documents": len(all_semantic_documents),
            "sentiment_documents": int((all_semantic_labels >= 0).sum()),
        },
    ]
)

group_summary



,group,semantic_documents,sentiment_documents
0,random,0,0
1,semantic_only,20000,0
2,semantic_sentiment,20000,20000
3,semantic_sentiment_unlabeled,70000,20000


In [ ]:
def make_initial_embedding():
    generator = torch.Generator().manual_seed(SEED)
    weights = torch.empty(TOTAL_VOCAB_SIZE, EMBED_DIM)
    weights.normal_(
        mean=0.0,
        std=EMBED_INIT_STD,
        generator=generator,
    )
    weights[PAD_ID].zero_()
    return weights


initial_embedding = make_initial_embedding()


class JointEmbeddingModel(nn.Module):
    def __init__(self, num_documents, initial_weights):
        super().__init__()
        document_generator = torch.Generator().manual_seed(
            SEED + 1
        )
        sentiment_generator = torch.Generator().manual_seed(
            SEED + 2
        )

        self.word_embeddings = nn.Embedding.from_pretrained(
            initial_weights.clone(),
            freeze=False,
            padding_idx=PAD_ID,
        )
        self.document_embeddings = nn.Embedding(
            num_documents,
            EMBED_DIM,
        )
        self.word_bias = nn.Embedding(
            TOTAL_VOCAB_SIZE,
            1,
            padding_idx=PAD_ID,
        )
        self.sentiment_head = nn.Linear(EMBED_DIM, 1)

        nn.init.normal_(
            self.document_embeddings.weight,
            mean=0.0,
            std=EMBED_INIT_STD,
            generator=document_generator,
        )
        nn.init.zeros_(self.word_bias.weight)
        nn.init.normal_(
            self.sentiment_head.weight,
            mean=0.0,
            std=EMBED_INIT_STD,
            generator=sentiment_generator,
        )
        nn.init.zeros_(self.sentiment_head.bias)


print("Initial embedding shape:", tuple(initial_embedding.shape))



Initial embedding shape: (8002, 50)


train embedding

In [ ]:
def build_batch_word_counts(
    documents,
    document_indices,
):
    batch_documents = [
        documents[index]
        for index in document_indices.tolist()
    ]

    vocabulary_ids = [
        document[document >= 2] - 2
        for document in batch_documents
    ]
    document_lengths = torch.tensor(
        [len(document) for document in vocabulary_ids],
        dtype=torch.long,
    )

    row_ids = torch.repeat_interleave(
        torch.arange(len(vocabulary_ids)),
        document_lengths,
    )
    word_ids = torch.cat(vocabulary_ids)
    flat_ids = row_ids * VOCAB_SIZE + word_ids

    return torch.bincount(
        flat_ids,
        minlength=len(vocabulary_ids) * VOCAB_SIZE,
    ).view(len(vocabulary_ids), VOCAB_SIZE).float()



In [ ]:
def train_embedding_group(
    group_name,
    documents,
    labels,
    use_sentiment,
):
    set_seed(SEED)

    model = JointEmbeddingModel(
        num_documents=len(documents),
        initial_weights=initial_embedding,
    ).to(device)
    initial_unk_vector = initial_embedding[UNK_ID].to(device)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.word_embeddings.parameters(),
                "weight_decay": WEIGHT_DECAY,
            },
            {
                "params": model.document_embeddings.parameters(),
                "weight_decay": WEIGHT_DECAY,
            },
            {
                "params": model.word_bias.parameters(),
                "weight_decay": 0.0,
            },
            {
                "params": model.sentiment_head.parameters(),
                "weight_decay": 0.0,
            },
        ],
        lr=LEARNING_RATE,
    )

    loader_generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        TensorDataset(
            torch.arange(len(documents)),
            labels,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        generator=loader_generator,
    )

    history = []
    best_training_loss = float("inf")
    best_patience_loss = float("inf")
    best_training_epoch = None
    best_word_weights = None
    best_sentiment_weight = None
    best_sentiment_bias = None
    epochs_without_improvement = 0
    stopped_early = False

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        total_semantic_nll = 0.0
        total_sentiment_nll = 0.0
        total_tokens = 0

        progress = tqdm(
            loader,
            desc=f"{group_name} | epoch {epoch}",
            leave=False,
        )

        for document_indices, batch_labels in progress:
            batch_word_counts = build_batch_word_counts(
                documents,
                document_indices,
            )
            batch_token_count = int(
                batch_word_counts.sum().item()
            )

            if batch_token_count == 0:
                continue

            document_indices = document_indices.to(device)
            batch_labels = batch_labels.to(device)
            batch_word_counts = batch_word_counts.to(device)

            document_vectors = model.document_embeddings(
                document_indices
            )
            word_vectors = model.word_embeddings.weight[2:]
            word_bias = model.word_bias.weight[2:].squeeze(-1)

            semantic_logits = (
                document_vectors @ word_vectors.T
                + word_bias
            )
            semantic_log_probs = F.log_softmax(
                semantic_logits,
                dim=1,
            )
            semantic_nll = -(
                batch_word_counts * semantic_log_probs
            ).sum()
            semantic_loss = (
                semantic_nll / batch_token_count
            )

            sentiment_nll = torch.zeros(
                (),
                device=device,
            )

            labelled_mask = batch_labels >= 0
            if use_sentiment and labelled_mask.any():
                labelled_counts = batch_word_counts[
                    labelled_mask
                ]
                labelled_labels = batch_labels[
                    labelled_mask
                ].float()

                positive_counts = (
                    labelled_counts
                    * labelled_labels.unsqueeze(1)
                ).sum(dim=0)
                negative_counts = (
                    labelled_counts
                    * (1.0 - labelled_labels).unsqueeze(1)
                ).sum(dim=0)

                sentiment_logits = model.sentiment_head(
                    word_vectors
                ).squeeze(-1)
                sentiment_nll = (
                    F.softplus(-sentiment_logits)
                    * positive_counts
                    + F.softplus(sentiment_logits)
                    * negative_counts
                ).sum()

            sentiment_loss = (
                sentiment_nll / batch_token_count
            )

            loss = (
                semantic_loss
                + SENTIMENT_WEIGHT * sentiment_loss
            )

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            with torch.no_grad():
                model.word_embeddings.weight[PAD_ID].zero_()
                model.word_embeddings.weight[UNK_ID].copy_(
                    initial_unk_vector
                )

            total_semantic_nll += semantic_nll.item()
            total_sentiment_nll += sentiment_nll.item()
            total_tokens += batch_token_count

        epoch_record = {
            "group": group_name,
            "epoch": epoch,
            "loss": (
                total_semantic_nll
                + SENTIMENT_WEIGHT * total_sentiment_nll
            ) / total_tokens,
            "semantic_loss": (
                total_semantic_nll / total_tokens
            ),
            "sentiment_loss": (
                total_sentiment_nll / total_tokens
            ),
        }
        history.append(epoch_record)

        current_loss = epoch_record["loss"]
        is_lowest_training_loss = (
            current_loss < best_training_loss
        )
        if is_lowest_training_loss:
            best_training_loss = current_loss
            best_training_epoch = epoch
            best_word_weights = (
                model.word_embeddings.weight
                .detach()
                .cpu()
                .clone()
            )

            if use_sentiment:
                best_sentiment_weight = (
                    model.sentiment_head.weight
                    .detach()
                    .cpu()
                    .clone()
                )
                best_sentiment_bias = (
                    model.sentiment_head.bias
                    .detach()
                    .cpu()
                    .clone()
                )

        is_meaningful_improvement = (
            current_loss
            < best_patience_loss
            - EARLY_STOPPING_MIN_DELTA
        )
        if is_meaningful_improvement:
            best_patience_loss = current_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{group_name} | epoch {epoch:02d} | "
            f"semantic={epoch_record['semantic_loss']:.4f} | "
            f"sentiment={epoch_record['sentiment_loss']:.4f} | "
            f"total={current_loss:.4f}"
            + (
                " | lowest training objective so far"
                if is_lowest_training_loss
                else ""
            )
            + (
                f" | early_stop={epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}"
            )
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stopped_early = True
            print(
                f"{group_name} | early stopping at epoch "
                f"{epoch:02d}: no improvement >= "
                f"{EARLY_STOPPING_MIN_DELTA:g} for "
                f"{EARLY_STOPPING_PATIENCE} epochs"
            )
            break

    print(
        f"{group_name} | selected best Embedding: "
        f"epoch {best_training_epoch:02d} "
        f"(loss={best_training_loss:.4f}); "
        f"trained {len(history)} epoch(s)"
    )

    result = {
        "weight": best_word_weights,
        "best_training_epoch": best_training_epoch,
        "best_training_loss": best_training_loss,
        "epochs_trained": len(history),
        "stopped_early": stopped_early,
        "sentiment_weight": best_sentiment_weight,
        "sentiment_bias": best_sentiment_bias,
    }

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, history



In [ ]:
embedding_results = {
    "random": {
        "weight": initial_embedding.clone(),
        "best_training_epoch": 0,
        "best_training_loss": None,
        "epochs_trained": 0,
        "stopped_early": False,
        "sentiment_weight": None,
        "sentiment_bias": None,
    }
}
history_rows = []

for group_name, spec in pretraining_specs.items():
    print("\n" + "=" * 70)
    print(group_name)
    print("=" * 70)

    group_result, group_history = train_embedding_group(
        group_name=group_name,
        documents=spec["documents"],
        labels=spec["labels"],
        use_sentiment=spec["use_sentiment"],
    )

    embedding_results[group_name] = group_result
    history_rows.extend(group_history)

history_df = pd.DataFrame(history_rows)
history_df


semantic_only


semantic_only | epoch 01 | semantic=8.9070 | sentiment=0.0000 | total=8.9070 | lowest training objective so far | early_stop=0/4


semantic_only | epoch 02 | semantic=8.7155 | sentiment=0.0000 | total=8.7155 | lowest training objective so far | early_stop=0/4


semantic_only | epoch 03 | semantic=8.2914 | sentiment=0.0000 | total=8.2914 | lowest training objective so far | early_stop=0/4


semantic_only | epoch 04 | semantic=7.8112 | sentiment=0.0000 | total=7.8112 | lowest training objective so far | early_stop=0/4


...

semantic_sentiment_unlabeled | epoch 27 | semantic=6.9838 | sentiment=0.1917 | total=7.1755 | lowest training objective so far | early_stop=0/4


semantic_sentiment_unlabeled | epoch 28 | semantic=6.9782 | sentiment=0.1917 | total=7.1699 | lowest training objective so far | early_stop=0/4


semantic_sentiment_unlabeled | epoch 29 | semantic=6.9733 | sentiment=0.1917 | total=7.1650 | lowest training objective so far | early_stop=0/4


semantic_sentiment_unlabeled | epoch 30 | semantic=6.9686 | sentiment=0.1917 | total=7.1603 | lowest training objective so far | early_stop=0/4
semantic_sentiment_unlabeled | selected best Embedding: epoch 30 (loss=7.1603); trained 30 epoch(s)


,group,epoch,loss,semantic_loss,sentiment_loss
0,semantic_only,1,8.907041,8.907041,0.000000
1,semantic_only,2,8.715535,8.715535,0.000000
2,semantic_only,3,8.291359,8.291359,0.000000
3,semantic_only,4,7.811232,7.811232,0.000000
4,semantic_only,5,7.701996,7.701996,0.000000
...,...,...,...,...,...
85,semantic_sentiment_unlabeled,26,7.181562,6.989874,0.191687
86,semantic_sentiment_unlabeled,27,7.175475,6.983792,0.191683
87,semantic_sentiment_unlabeled,28,7.169949,6.978248,0.191701
88,semantic_sentiment_unlabeled,29,7.164958,6.973272,0.191686


In [ ]:
#save
from pathlib import Path

EMBEDDING_SAVE_PATH = Path("./model/four_embedding_results.pt")

GROUP_NAMES = [
    "random",
    "semantic_only",
    "semantic_sentiment",
    "semantic_sentiment_unlabeled",
]

checkpoint = {
    "embedding_results": {
        group_name: {
            key: (
                value.detach().cpu().clone()
                if torch.is_tensor(value)
                else value
            )
            for key, value in embedding_results[group_name].items()
        }
        for group_name in GROUP_NAMES
    },
    "id_to_token": id_to_token,
    "token_to_id": token_to_id,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,

    "total_vocab_size": TOTAL_VOCAB_SIZE,
    "embed_dim": EMBED_DIM,
    "seed": SEED,
}

torch.save(checkpoint, EMBEDDING_SAVE_PATH)

print(f"Saved to: {EMBEDDING_SAVE_PATH.resolve()}")

for group_name in GROUP_NAMES:
    result = checkpoint["embedding_results"][group_name]
    print(
        f"{group_name:34s} "
        f"shape={tuple(result['weight'].shape)} | "
        f"best_epoch={result['best_training_epoch']}"
    )

Saved to: C:\Users\lulu\Desktop\9444\ASSISMENT\model\four_embedding_results.pt
random                             shape=(8002, 50) | best_epoch=0
semantic_only                      shape=(8002, 50) | best_epoch=30
semantic_sentiment                 shape=(8002, 50) | best_epoch=30
semantic_sentiment_unlabeled       shape=(8002, 50) | best_epoch=30


In [ ]:
for group_name, result in embedding_results.items():
    weights = result["weight"]

    distance_from_initial = torch.linalg.vector_norm(
        weights - initial_embedding
    ).item()

    print(
        f"{group_name:34s} "
        f"epoch={result['best_training_epoch']:02d} "
        f"trained={result['epochs_trained']:02d} "
        f"early_stopped={result['stopped_early']} "
        f"shape={tuple(weights.shape)} "
        f"distance_from_initial={distance_from_initial:.4f}"
    )



random                             epoch=00 trained=00 early_stopped=False shape=(8002, 50) distance_from_initial=0.0000
semantic_only                      epoch=30 trained=30 early_stopped=False shape=(8002, 50) distance_from_initial=462.1321
semantic_sentiment                 epoch=30 trained=30 early_stopped=False shape=(8002, 50) distance_from_initial=355.8790
semantic_sentiment_unlabeled       epoch=30 trained=30 early_stopped=False shape=(8002, 50) distance_from_initial=592.1082


### Inspect learned Embeddings

In [ ]:
def nearest_words(
    group_name,
    query_word,
    top_k=8,
):
    if query_word not in token_to_id:
        return []

    weights = embedding_results[group_name]["weight"]
    normalised = F.normalize(
        weights[2:],
        p=2,
        dim=1,
    )

    query_id = token_to_id[query_word]
    query_vector = F.normalize(
        weights[query_id].unsqueeze(0),
        p=2,
        dim=1,
    )

    similarities = normalised @ query_vector.squeeze(0)
    similarities[query_id - 2] = -torch.inf

    neighbour_positions = torch.topk(
        similarities,
        k=top_k,
    ).indices

    return [
        id_to_token[position.item() + 2]
        for position in neighbour_positions
    ]


for query_word in ["excellent", "awful", "boring"]:
    if query_word not in token_to_id:
        continue

    print(f"\nQuery: {query_word}")
    for group_name in sorted(embedding_results):
        print(
            f"{group_name:34s}",
            nearest_words(group_name, query_word),
        )




Query: excellent
random                             ['wondered', 'phil', 'window', 'gritty', 'obnoxious', 'integrity', 'nights', 'cooper']
semantic_only                      ['superb', 'perfect', 'rate', 'outstanding', 'highly', 'strong', 'impressed', 'underrated']
semantic_sentiment                 ['superb', 'outstanding', 'brilliant', 'underrated', 'rate', 'fantastic', 'highly', 'performances']
semantic_sentiment_unlabeled       ['outstanding', 'superb', 'exceptional', 'superbly', 'flawless', 'fantastic', 'highly', 'recommended']

Query: awful
random                             ['clown', 'challenges', 'goals', 'juan', 'pizza', 'millions', 'sole', 'tramp']
semantic_only                      ['waste', 'redeeming', 'lousy', 'worst', 'atrocious', 'pathetic', 'terrible', 'horrible']
semantic_sentiment                 ['terrible', 'waste', 'amateur', 'worst', 'dreadful', 'wasted', 'poorly', 'costs']
semantic_sentiment_unlabeled       ['terrible', 'avoid', 'lousy', 'dreadful', 'laughable'

## CNN

simple one just for result

In [ ]:
# CNN hyperparameters
CNN_MAX_LENGTH = 1000
CNN_FILTERS = 1000
CNN_KERNEL_SIZE = 4
CNN_DROPOUT = 0.5
CNN_BATCH_SIZE = 128
CNN_MAX_EPOCHS = 15
CNN_EARLY_STOPPING_PATIENCE = 3
CNN_EARLY_STOPPING_MIN_DELTA = 1e-4
CNN_LEARNING_RATE = 1e-3
CNN_WEIGHT_DECAY = 1e-4
CNN_GRAD_CLIP = 5.0
CNN_FREEZE_EMBEDDING = False
CNN_NUM_WORKERS = 0

CNN_GROUPS = [
    "random",
    "semantic_only",
    "semantic_sentiment",
    "semantic_sentiment_unlabeled",
]

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
from torch.utils.data import Dataset


valid_documents = encode_documents(
    valid_dataset["text"],
    "Encoding validation reviews",
)
test_documents = encode_documents(
    test_dataset["text"],
    "Encoding test reviews",
)

valid_labels = torch.tensor(
    valid_dataset["label"],
    dtype=torch.long,
)
test_labels = torch.tensor(
    test_dataset["label"],
    dtype=torch.long,
)


class ReviewDataset(Dataset):
    def __init__(self, documents, labels):
        self.documents = documents
        self.labels = labels

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, index):
        return self.documents[index], self.labels[index]


def collate_reviews(batch):
    documents, labels = zip(*batch)
    documents = [
        document[:CNN_MAX_LENGTH]
        for document in documents
    ]
    lengths = torch.tensor(
        [len(document) for document in documents],
        dtype=torch.long,
    )
    padded_length = max(
        CNN_KERNEL_SIZE,
        int(lengths.max()),
    )
    input_ids = torch.full(
        (len(documents), padded_length),
        PAD_ID,
        dtype=torch.long,
    )

    for row, document in enumerate(documents):
        input_ids[row, :len(document)] = document

    labels = torch.tensor(
        [int(label) for label in labels],
        dtype=torch.long,
    )
    return input_ids, lengths, labels


train_review_dataset = ReviewDataset(
    train_documents,
    train_labels,
)
valid_review_dataset = ReviewDataset(
    valid_documents,
    valid_labels,
)
test_review_dataset = ReviewDataset(
    test_documents,
    test_labels,
)


def make_cnn_loader(dataset, shuffle):
    generator = (
        torch.Generator().manual_seed(SEED)
        if shuffle
        else None
    )
    return DataLoader(
        dataset,
        batch_size=CNN_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=CNN_NUM_WORKERS,
        collate_fn=collate_reviews,
        pin_memory=(device.type == "cuda"),
        generator=generator,
    )


valid_loader = make_cnn_loader(
    valid_review_dataset,
    shuffle=False,
)
test_loader = make_cnn_loader(
    test_review_dataset,
    shuffle=False,
)

print(
    f"CNN data: train={len(train_review_dataset):,}, "
    f"validation={len(valid_review_dataset):,}, "
    f"test={len(test_review_dataset):,}"
)


Encoding test reviews: 100%|██████████| 25000/25000 [00:02<00:00, 12187.09it/s]


CNN data: train=20,000, validation=5,000, test=25,000


In [ ]:
class SimpleTextCNN(nn.Module):
    def __init__(self, embedding_weights):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights.clone(),
            freeze=CNN_FREEZE_EMBEDDING,
            padding_idx=PAD_ID,
        )
        self.convolution = nn.Conv1d(
            in_channels=EMBED_DIM,
            out_channels=CNN_FILTERS,
            kernel_size=CNN_KERNEL_SIZE,
        )
        self.dropout = nn.Dropout(CNN_DROPOUT)
        self.classifier = nn.Linear(CNN_FILTERS, 1)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        embedded = embedded.transpose(1, 2)

        features = F.relu(
            self.convolution(embedded)
        )

        valid_steps = (
            lengths - CNN_KERNEL_SIZE + 1
        ).clamp(min=1)
        positions = torch.arange(
            features.size(2),
            device=features.device,
        ).unsqueeze(0)
        padding_mask = positions >= valid_steps.unsqueeze(1)
        features = features.masked_fill(
            padding_mask.unsqueeze(1),
            -torch.inf,
        )

        pooled = features.max(dim=2).values
        logits = self.classifier(
            self.dropout(pooled)
        )
        return logits.squeeze(1)


In [ ]:
def binary_metrics(labels, predictions):
    true_positive = int(
        ((labels == 1) & (predictions == 1)).sum()
    )
    true_negative = int(
        ((labels == 0) & (predictions == 0)).sum()
    )
    false_positive = int(
        ((labels == 0) & (predictions == 1)).sum()
    )
    false_negative = int(
        ((labels == 1) & (predictions == 0)).sum()
    )

    total = len(labels)
    accuracy = (true_positive + true_negative) / total
    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


@torch.inference_mode()
def evaluate_cnn(model, loader):
    model.eval()
    criterion = nn.BCEWithLogitsLoss(reduction="sum")

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        logits = model(input_ids, lengths)
        total_loss += criterion(
            logits,
            labels.float(),
        ).item()

        predictions = (logits >= 0).long()
        all_labels.append(labels.cpu())
        all_predictions.append(predictions.cpu())

    all_labels = torch.cat(all_labels)
    all_predictions = torch.cat(all_predictions)

    metrics = binary_metrics(
        all_labels,
        all_predictions,
    )
    metrics["loss"] = total_loss / len(all_labels)
    return metrics


In [ ]:
def train_cnn_group(group_name):
    set_seed(SEED)

    train_loader = make_cnn_loader(
        train_review_dataset,
        shuffle=True,
    )
    model = SimpleTextCNN(
        embedding_results[group_name]["weight"]
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNN_LEARNING_RATE,
        weight_decay=CNN_WEIGHT_DECAY,
    )
    criterion = nn.BCEWithLogitsLoss()

    best_validation_f1 = -float("inf")
    best_patience_f1 = -float("inf")
    best_epoch = None
    best_state = None
    epochs_without_improvement = 0
    stopped_early = False
    history = []

    for epoch in range(1, CNN_MAX_EPOCHS + 1):
        model.train()
        total_train_loss = 0.0
        total_train_examples = 0

        progress = tqdm(
            train_loader,
            desc=f"{group_name} | CNN epoch {epoch}",
            leave=False,
        )

        for input_ids, lengths, labels in progress:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels.float())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CNN_GRAD_CLIP,
            )
            optimizer.step()

            batch_size = len(labels)
            total_train_loss += loss.item() * batch_size
            total_train_examples += batch_size

        validation_metrics = evaluate_cnn(
            model,
            valid_loader,
        )
        train_loss = (
            total_train_loss / total_train_examples
        )
        current_f1 = validation_metrics["f1"]

        history.append(
            {
                "group": group_name,
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_metrics["loss"],
                "validation_accuracy": validation_metrics["accuracy"],
                "validation_precision": validation_metrics["precision"],
                "validation_recall": validation_metrics["recall"],
                "validation_f1": current_f1,
            }
        )

        is_best = current_f1 > best_validation_f1
        if is_best:
            best_validation_f1 = current_f1
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

        is_meaningful_improvement = (
            current_f1
            > best_patience_f1
            + CNN_EARLY_STOPPING_MIN_DELTA
        )
        if is_meaningful_improvement:
            best_patience_f1 = current_f1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{group_name:34s} | epoch {epoch:02d} | "
            f"train loss={train_loss:.4f} | "
            f"val acc={validation_metrics['accuracy']:.4f} | "
            f"val P={validation_metrics['precision']:.4f} | "
            f"val R={validation_metrics['recall']:.4f} | "
            f"val F1={current_f1:.4f}"
            + (" | best" if is_best else "")
            + (
                f" | early_stop={epochs_without_improvement}"
                f"/{CNN_EARLY_STOPPING_PATIENCE}"
            )
        )

        if (
            epochs_without_improvement
            >= CNN_EARLY_STOPPING_PATIENCE
        ):
            stopped_early = True
            break

    model.load_state_dict(best_state)
    validation_metrics = evaluate_cnn(model, valid_loader)
    test_metrics = evaluate_cnn(model, test_loader)

    result = {
        "group": group_name,
        "best_epoch": best_epoch,
        "epochs_trained": len(history),
        "early_stopped": stopped_early,
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_precision": validation_metrics["precision"],
        "validation_recall": validation_metrics["recall"],
        "validation_f1": validation_metrics["f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
    }

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, history


In [ ]:
cnn_results = []
cnn_history_rows = []

for group_name in CNN_GROUPS:
    print("\n" + "=" * 70)
    print(group_name)
    print("=" * 70)

    group_result, group_history = train_cnn_group(
        group_name
    )
    cnn_results.append(group_result)
    cnn_history_rows.extend(group_history)

    print(
        f"{group_name} | selected CNN epoch "
        f"{group_result['best_epoch']:02d} | "
        f"test acc={group_result['test_accuracy']:.4f} | "
        f"test P={group_result['test_precision']:.4f} | "
        f"test R={group_result['test_recall']:.4f} | "
        f"test F1={group_result['test_f1']:.4f}"
    )

cnn_results_df = pd.DataFrame(cnn_results)
cnn_history_df = pd.DataFrame(cnn_history_rows)
cnn_results_df.round(4)



random


random                             | epoch 01 | train loss=0.5272 | val acc=0.8436 | val P=0.8420 | val R=0.8460 | val F1=0.8440 | best | early_stop=0/3


random                             | epoch 02 | train loss=0.2832 | val acc=0.8542 | val P=0.8619 | val R=0.8436 | val F1=0.8526 | best | early_stop=0/3


random                             | epoch 03 | train loss=0.1948 | val acc=0.8532 | val P=0.8496 | val R=0.8584 | val F1=0.8540 | best | early_stop=0/3


random                             | epoch 04 | train loss=0.1307 | val acc=0.8496 | val P=0.8559 | val R=0.8408 | val F1=0.8483 | early_stop=1/3


...

semantic_sentiment_unlabeled       | epoch 03 | train loss=0.2362 | val acc=0.8858 | val P=0.8731 | val R=0.9028 | val F1=0.8877 | best | early_stop=0/3


semantic_sentiment_unlabeled       | epoch 04 | train loss=0.2064 | val acc=0.8872 | val P=0.8925 | val R=0.8804 | val F1=0.8864 | early_stop=1/3


semantic_sentiment_unlabeled       | epoch 05 | train loss=0.1887 | val acc=0.8852 | val P=0.8889 | val R=0.8804 | val F1=0.8846 | early_stop=2/3


semantic_sentiment_unlabeled       | epoch 06 | train loss=0.1692 | val acc=0.8850 | val P=0.8901 | val R=0.8784 | val F1=0.8842 | early_stop=3/3
semantic_sentiment_unlabeled | selected CNN epoch 03 | test acc=0.8863 | test P=0.8722 | test R=0.9053 | test F1=0.8884


,group,best_epoch,epochs_trained,early_stopped,validation_accuracy,validation_precision,validation_recall,validation_f1,test_accuracy,test_precision,test_recall,test_f1
0,random,3,6,True,0.8532,0.8496,0.8584,0.8540,0.8544,0.8534,0.8559,0.8547
1,semantic_only,4,7,True,0.8880,0.8783,0.9008,0.8894,0.8849,0.8752,0.8978,0.8863
2,semantic_sentiment,3,6,True,0.8832,0.8662,0.9064,0.8858,0.8772,0.8618,0.8985,0.8798
3,semantic_sentiment_unlabeled,3,6,True,0.8858,0.8731,0.9028,0.8877,0.8863,0.8722,0.9053,0.8884


## Compare

In [ ]:
random_result = cnn_results_df.loc[
    cnn_results_df["group"] == "random"
].iloc[0]

cnn_comparison_df = cnn_results_df[
    [
        "group",
        "best_epoch",
        "epochs_trained",
        "validation_f1",
        "test_accuracy",
        "test_precision",
        "test_recall",
        "test_f1",
    ]
].copy()

cnn_comparison_df["accuracy_vs_random"] = (
    cnn_comparison_df["test_accuracy"]
    - random_result["test_accuracy"]
)
cnn_comparison_df["f1_vs_random"] = (
    cnn_comparison_df["test_f1"]
    - random_result["test_f1"]
)
cnn_comparison_df["f1_rank"] = (
    cnn_comparison_df["test_f1"]
    .rank(method="min", ascending=False)
    .astype(int)
)
cnn_comparison_df = cnn_comparison_df.sort_values(
    ["f1_rank", "group"]
).reset_index(drop=True)

best_result = cnn_comparison_df.iloc[0]
print(
    f"Best test F1: {best_result['group']} "
    f"({best_result['test_f1']:.4f}, "
    f"change from random "
    f"{best_result['f1_vs_random']:+.4f})"
)

cnn_comparison_df.round(4)


Best test F1: semantic_sentiment_unlabeled (0.8884, change from random +0.0338)


,group,best_epoch,epochs_trained,validation_f1,test_accuracy,test_precision,test_recall,test_f1,accuracy_vs_random,f1_vs_random,f1_rank
0,semantic_sentiment_unlabeled,3,6,0.8877,0.8863,0.8722,0.9053,0.8884,0.0319,0.0338,1
1,semantic_only,4,7,0.8894,0.8849,0.8752,0.8978,0.8863,0.0304,0.0317,2
2,semantic_sentiment,3,6,0.8858,0.8772,0.8618,0.8985,0.8798,0.0228,0.0251,3
3,random,3,6,0.8540,0.8544,0.8534,0.8559,0.8547,0.0000,0.0000,4
